# Experiment 7: feature cleaning exploration

This is a read-only audit. It identifies categorical standardisation opportunities, potential numeric outliers, and feature/target associations before modelling changes are made. It exports the evidence for review and does **not** change preprocessing.py.

In [10]:
from pathlib import Path
import os
import sys
import re
import numpy as np
import pandas as pd

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from preprocessing import RentalPreprocessor, load_datasets, normalize_key

_, datasets = load_datasets(project_root / 'data')
train_raw = datasets['train'].copy()
test_raw = datasets['test'].copy()
hdb_raw = datasets['hdb'].copy()
print(f'Train: {train_raw.shape}; test: {test_raw.shape}; HDB lookup: {hdb_raw.shape}')


Train: (150000, 11); test: (50000, 10); HDB lookup: (9464, 11)


## 1. Categorical feature standardisation

Compare increasingly conservative normalisation rules across rental and HDB categorical fields. Review collisions and train/test coverage before deciding which formatting-only changes belong in shared preprocessing. Legitimate unseen test categories are retained; this audit never removes them.

In [11]:
# Each normaliser is deliberately more aggressive than the preceding one.
# The audit compares their effects; it does not apply any of them to model data.
def as_text(values):
    return values.fillna('<MISSING>').astype(str)

def case_whitespace(values):
    return as_text(values).str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def separator_fold(values):
    return case_whitespace(values).str.replace(r'[-_/]+', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

def alphanumeric_fold(values):
    return case_whitespace(values).str.replace(r'[^a-z0-9]+', '', regex=True)

NORMALISERS = {
    'raw': as_text,
    'case_whitespace': case_whitespace,
    'separator_fold': separator_fold,
    'alphanumeric_fold': alphanumeric_fold,
}


In [12]:
MAIN_CATEGORICAL_COLUMNS = ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FURNISHED']
main_columns = [column for column in MAIN_CATEGORICAL_COLUMNS if column in train_raw.columns]
combined_main = pd.concat([
    train_raw.assign(_source='train'),
    test_raw.assign(_source='test'),
], ignore_index=True)

summary_rows = []
collision_rows = []
for column in main_columns:
    values = combined_main[column]
    for normaliser_name, normaliser in NORMALISERS.items():
        normalised = normaliser(values)
        raw_variants_per_key = pd.DataFrame({'raw': as_text(values), 'key': normalised}).groupby('key')['raw'].nunique()
        summary_rows.append({
            'dataset': 'main_train_and_test', 'column': column, 'normaliser': normaliser_name,
            'unique_values': normalised.nunique(dropna=False),
            'keys_with_multiple_raw_variants': int((raw_variants_per_key > 1).sum()),
            'rows_in_colliding_keys': int(normalised.isin(raw_variants_per_key[raw_variants_per_key > 1].index).sum()),
        })
        if normaliser_name != 'raw':
            variants = pd.DataFrame({'raw': as_text(values), 'key': normalised}).drop_duplicates()
            for key, group in variants.groupby('key'):
                raw_values = sorted(group['raw'].tolist())
                if len(raw_values) > 1:
                    collision_rows.append({
                        'dataset': 'main_train_and_test', 'column': column, 'normaliser': normaliser_name,
                        'canonical_key': key, 'raw_variants': ' | '.join(raw_values[:12]),
                        'variant_count': len(raw_values),
                    })

main_summary = pd.DataFrame(summary_rows)
main_collisions = pd.DataFrame(collision_rows).sort_values(['column', 'normaliser', 'variant_count'], ascending=[True, True, False])
display(main_summary.pivot(index='column', columns='normaliser', values='unique_values'))
display(main_collisions.head(100))


normaliser,alphanumeric_fold,case_whitespace,raw,separator_fold
column,,,,
BLOCK,2731,2731,2731,2731
FLAT_MODEL,20,20,20,20
FLAT_TYPE,6,11,11,6
FURNISHED,1,1,1,1
STREET,576,576,1150,576
TOWN,26,26,26,26


,dataset,column,normaliser,canonical_key,raw_variants,variant_count
1727,main_train_and_test,FLAT_TYPE,alphanumeric_fold,1room,1 room | 1-room,2
1728,main_train_and_test,FLAT_TYPE,alphanumeric_fold,2room,2 room | 2-room,2
1729,main_train_and_test,FLAT_TYPE,alphanumeric_fold,3room,3 room | 3-room,2
1730,main_train_and_test,FLAT_TYPE,alphanumeric_fold,4room,4 room | 4-room,2
1731,main_train_and_test,FLAT_TYPE,alphanumeric_fold,5room,5 room | 5-room,2
...,...,...,...,...,...,...
1233,main_train_and_test,STREET,alphanumeric_fold,bukitbatokstreet41,Bukit Batok Street 41 | bukit batok street 41,2
1234,main_train_and_test,STREET,alphanumeric_fold,bukitbatokstreet51,Bukit Batok Street 51 | bukit batok street 51,2
1235,main_train_and_test,STREET,alphanumeric_fold,bukitbatokstreet52,Bukit Batok Street 52 | bukit batok street 52,2
1236,main_train_and_test,STREET,alphanumeric_fold,bukitbatokwestavenue2,Bukit Batok West Avenue 2 | bukit batok west a...,2


In [13]:
# Check whether normalisation changes how many test categories are unseen during training.
coverage_rows = []
for column in main_columns:
    for normaliser_name, normaliser in NORMALISERS.items():
        train_values = set(normaliser(train_raw[column]))
        test_values = normaliser(test_raw[column])
        unseen_mask = ~test_values.isin(train_values)
        coverage_rows.append({
            'column': column, 'normaliser': normaliser_name,
            'train_unique': len(train_values), 'test_unique': test_values.nunique(),
            'unseen_test_categories': test_values[unseen_mask].nunique(),
            'test_rows_with_unseen_category': int(unseen_mask.sum()),
            'test_row_share_unseen': float(unseen_mask.mean()),
        })
coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report.sort_values(['column', 'normaliser']))

# Show exact raw train/test variants for the columns where separator folding improves coverage.
for column in main_columns:
    raw_unseen = coverage_report[(coverage_report['column'] == column) & (coverage_report['normaliser'] == 'raw')].iloc[0]['test_rows_with_unseen_category']
    folded_unseen = coverage_report[(coverage_report['column'] == column) & (coverage_report['normaliser'] == 'separator_fold')].iloc[0]['test_rows_with_unseen_category']
    if folded_unseen < raw_unseen:
        print(f'\n{column}: separator folding removes {int(raw_unseen - folded_unseen):,} unseen test rows')
        print(sorted(set(as_text(test_raw[column])) - set(as_text(train_raw[column])))[:30])


,column,normaliser,train_unique,test_unique,unseen_test_categories,test_rows_with_unseen_category,test_row_share_unseen
7,BLOCK,alphanumeric_fold,2720,2660,11,36,0.00072
5,BLOCK,case_whitespace,2720,2660,11,36,0.00072
4,BLOCK,raw,2720,2660,11,36,0.00072
6,BLOCK,separator_fold,2720,2660,11,36,0.00072
19,FLAT_MODEL,alphanumeric_fold,20,19,0,0,0.00000
17,FLAT_MODEL,case_whitespace,20,19,0,0,0.00000
16,FLAT_MODEL,raw,20,19,0,0,0.00000
18,FLAT_MODEL,separator_fold,20,19,0,0,0.00000
15,FLAT_TYPE,alphanumeric_fold,6,6,0,0,0.00000
13,FLAT_TYPE,case_whitespace,11,6,0,0,0.00000



STREET: separator folding removes 6 unseen test rows
['Ang Mo Kio Street 23', 'Bukit Batok Street 41', 'Jurong West Street 51', 'Yishun Street 42']


In [14]:
# Audit every HDB text column. POSTAL_CODE is an identifier, so inspect it for formatting only;
# it should not be one-hot encoded as an ordinary category without a separate modelling decision.
HDB_CATEGORICAL_COLUMNS = [
    column for column in hdb_raw.columns
    if pd.api.types.is_string_dtype(hdb_raw[column]) or hdb_raw[column].dtype == object
]
hdb_summary_rows = []
hdb_collision_rows = []
for column in HDB_CATEGORICAL_COLUMNS:
    for normaliser_name, normaliser in NORMALISERS.items():
        normalised = normaliser(hdb_raw[column])
        variants = pd.DataFrame({'raw': as_text(hdb_raw[column]), 'key': normalised}).drop_duplicates()
        raw_variants_per_key = variants.groupby('key')['raw'].nunique()
        hdb_summary_rows.append({
            'dataset': 'hdb_lookup', 'column': column, 'normaliser': normaliser_name,
            'unique_values': normalised.nunique(),
            'keys_with_multiple_raw_variants': int((raw_variants_per_key > 1).sum()),
        })
        if normaliser_name != 'raw':
            for key, group in variants.groupby('key'):
                raw_values = sorted(group['raw'].tolist())
                if len(raw_values) > 1:
                    hdb_collision_rows.append({
                        'column': column, 'normaliser': normaliser_name, 'canonical_key': key,
                        'raw_variants': ' | '.join(raw_values[:12]), 'variant_count': len(raw_values),
                    })
hdb_summary = pd.DataFrame(hdb_summary_rows)
hdb_collisions = pd.DataFrame(hdb_collision_rows, columns=['column', 'normaliser', 'canonical_key', 'raw_variants', 'variant_count'])
display(hdb_summary.pivot(index='column', columns='normaliser', values='unique_values'))
display(hdb_collisions.head(100))

def location_coverage(normaliser_name, normaliser):
    listings = pd.concat([train_raw[['BLOCK', 'STREET']], test_raw[['BLOCK', 'STREET']]], ignore_index=True).copy()
    lookup = hdb_raw[['BLOCK', 'STREET']].copy()
    listings['block_key'] = normaliser(listings['BLOCK'])
    listings['street_key'] = normaliser(listings['STREET'])
    lookup['block_key'] = normaliser(lookup['BLOCK'])
    lookup['street_key'] = normaliser(lookup['STREET'])
    lookup_keys = pd.MultiIndex.from_frame(lookup[['block_key', 'street_key']].drop_duplicates())
    listing_keys = pd.MultiIndex.from_frame(listings[['block_key', 'street_key']])
    matched = listing_keys.isin(lookup_keys)
    return {
        'normaliser': normaliser_name, 'listing_rows': len(listings),
        'matched_rows': int(matched.sum()), 'match_rate': float(matched.mean()),
        'unmatched_unique_locations': int(pd.MultiIndex.from_frame(listings.loc[~matched, ['block_key', 'street_key']]).nunique()),
    }

location_report = pd.DataFrame([location_coverage(name, function) for name, function in NORMALISERS.items()])
display(location_report)
print('`case_whitespace` is equivalent to the current preprocessing.normalize_key behaviour for these fields.')


normaliser,alphanumeric_fold,case_whitespace,raw,separator_fold
column,,,,
BLOCK,2735,2735,2735,2735
PLANNING_AREA,31,31,31,31
POSTAL_CODE,9180,9180,9180,9180
REGION,5,5,5,5
STREET,576,576,576,576
SUBZONE,156,156,156,156
TOWN,26,26,26,26


,column,normaliser,canonical_key,raw_variants,variant_count


,normaliser,listing_rows,matched_rows,match_rate,unmatched_unique_locations
0,raw,200000,158737,0.793685,8565
1,case_whitespace,200000,200000,1.000000,0
2,separator_fold,200000,200000,1.000000,0
3,alphanumeric_fold,200000,200000,1.000000,0


`case_whitespace` is equivalent to the current preprocessing.normalize_key behaviour for these fields.


In [15]:
# FLOOR_AREA_SQM is numeric, not categorical. This only checks whether it needs cleaning/coercion.
area_rows = []
for source_name, frame in [('train', train_raw), ('test', test_raw)]:
    original = frame['FLOOR_AREA_SQM']
    numeric = pd.to_numeric(original, errors='coerce')
    area_rows.append({
        'source': source_name, 'rows': len(frame), 'null_or_non_numeric': int(numeric.isna().sum()),
        'min_sqm': numeric.min(), 'max_sqm': numeric.max(),
        'non_positive_values': int((numeric <= 0).sum()),
    })
area_report = pd.DataFrame(area_rows)
display(area_report)


,source,rows,null_or_non_numeric,min_sqm,max_sqm,non_positive_values
0,train,150000,0,31.0,215.0,0
1,test,50000,0,31.0,215.0,0


## 2. Detecting and reviewing numeric outliers

Flag unusual numeric values with IQR fences and robust z-scores. These are review candidates, not rows to remove automatically: unusually large flats, long leases, or high rents may be valid and especially relevant in future periods.

In [16]:
TARGET = 'MONTHLY_RENT'
outlier_frame = train_raw.copy()
approval_date = pd.to_datetime(outlier_frame['RENT_APPROVAL_DATE'] + '-01', errors='coerce')
outlier_frame['APPROVAL_YEAR'] = approval_date.dt.year
outlier_frame['LEASE_AGE_AT_APPROVAL'] = outlier_frame['APPROVAL_YEAR'] - pd.to_numeric(outlier_frame['LEASE_COMMENCE_DATE'], errors='coerce')

OUTLIER_COLUMNS = ['FLOOR_AREA_SQM', 'LEASE_COMMENCE_DATE', 'LEASE_AGE_AT_APPROVAL', TARGET]
outlier_rows = []
outlier_examples = []
for column in OUTLIER_COLUMNS:
    values = pd.to_numeric(outlier_frame[column], errors='coerce').dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    median = values.median()
    mad = (values - median).abs().median()
    robust_z = 0.6745 * (values - median).abs() / mad if mad else pd.Series(0.0, index=values.index)
    iqr_mask = (values < lower_fence) | (values > upper_fence)
    robust_mask = robust_z > 3.5
    flagged_index = values.index[iqr_mask | robust_mask]
    outlier_rows.append({
        'feature': column, 'non_null_rows': len(values), 'min': values.min(), 'max': values.max(),
        'q1': q1, 'median': median, 'q3': q3, 'iqr_lower_fence': lower_fence,
        'iqr_upper_fence': upper_fence, 'iqr_flagged_rows': int(iqr_mask.sum()),
        'robust_z_flagged_rows': int(robust_mask.sum()), 'flagged_by_either_rule': len(flagged_index),
    })
    examples = outlier_frame.loc[flagged_index, ['RENT_APPROVAL_DATE', 'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', column]].copy()
    examples.insert(0, 'feature', column)
    examples.rename(columns={column: 'value'}, inplace=True)
    outlier_examples.append(examples.sort_values('value').head(10))
    outlier_examples.append(examples.sort_values('value').tail(10))

outlier_summary = pd.DataFrame(outlier_rows)
outlier_examples = pd.concat(outlier_examples, ignore_index=True).drop_duplicates()
display(outlier_summary)
display(outlier_examples.sort_values(['feature', 'value']).head(40))


,feature,non_null_rows,min,max,q1,median,q3,iqr_lower_fence,iqr_upper_fence,iqr_flagged_rows,robust_z_flagged_rows,flagged_by_either_rule
0,FLOOR_AREA_SQM,150000,31.0,215.0,68.0,92.0,110.0,5.0,173.0,130,22,130
1,LEASE_COMMENCE_DATE,150000,1966.0,2022.0,1981.0,1988.0,2001.0,1951.0,2031.0,0,0,0
2,LEASE_AGE_AT_APPROVAL,150000,-1.0,59.0,22.0,34.0,42.0,-8.0,72.0,0,0,0
3,MONTHLY_RENT,150000,300.0,7600.0,2100.0,2700.0,3200.0,450.0,4850.0,752,61,752


,feature,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,value
0,FLOOR_AREA_SQM,2023-12,woodlands,141,marsiling road,executive,174.0
1,FLOOR_AREA_SQM,2023-07,woodlands,135,marsiling road,executive,174.0
2,FLOOR_AREA_SQM,2021-05,woodlands,135,marsiling road,executive,174.0
3,FLOOR_AREA_SQM,2021-06,woodlands,135,marsiling road,executive,174.0
4,FLOOR_AREA_SQM,2024-04,woodlands,141,marsiling road,executive,174.0
5,FLOOR_AREA_SQM,2021-04,woodlands,141,marsiling road,executive,174.0
6,FLOOR_AREA_SQM,2024-07,woodlands,135,marsiling road,executive,174.0
7,FLOOR_AREA_SQM,2021-07,woodlands,141,marsiling road,executive,174.0
8,FLOOR_AREA_SQM,2022-04,woodlands,307,woodlands avenue 1,executive,176.0
9,FLOOR_AREA_SQM,2022-09,woodlands,849,woodlands street 82,executive,176.0


## 3. Input-feature correlation and target association

Measure every numeric and categorical feature returned by shared preprocessing. This includes the HDB, proximity, COE, stock, and calendar features actually supplied to models. High-cardinality fields can look strongly associated in-sample, so use this to prioritise feature experiments and confirm every decision with chronological validation.

In [17]:
TARGET = 'MONTHLY_RENT'
# Analyse the exact columns supplied to downstream models by preprocessing.py.
# The preprocessor is fitted only on training data, then applied to test for coverage checks.
association_preprocessor = RentalPreprocessor(datasets)
association_frame = association_preprocessor.fit_transform(train_raw)
association_test_frame = association_preprocessor.transform(test_raw)
NUMERIC_INPUT_COLUMNS = association_frame.select_dtypes(include=['number', 'bool']).columns.drop(TARGET).tolist()
print(f'Analysing {len(NUMERIC_INPUT_COLUMNS)} numeric and {association_frame.shape[1] - len(NUMERIC_INPUT_COLUMNS) - 1} categorical preprocessor features.')
numeric_data = association_frame[NUMERIC_INPUT_COLUMNS + [TARGET]].apply(pd.to_numeric, errors='coerce')
numeric_target_associations = pd.DataFrame({
    'feature': NUMERIC_INPUT_COLUMNS,
    'pearson_with_rent': [numeric_data[column].corr(numeric_data[TARGET], method='pearson') for column in NUMERIC_INPUT_COLUMNS],
    'spearman_with_rent': [numeric_data[column].corr(numeric_data[TARGET], method='spearman') for column in NUMERIC_INPUT_COLUMNS],
    'non_null_rows': [numeric_data[[column, TARGET]].dropna().shape[0] for column in NUMERIC_INPUT_COLUMNS],
}).sort_values('spearman_with_rent', key=lambda values: values.abs(), ascending=False)
# Include the target in the matrix so each feature-to-rent correlation is visible in one view.
numeric_feature_correlation = numeric_data[NUMERIC_INPUT_COLUMNS + [TARGET]].corr(method='spearman')

CATEGORICAL_INPUT_COLUMNS = association_frame.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
categorical_rows = []
rent_variance_total = ((association_frame[TARGET] - association_frame[TARGET].mean()) ** 2).sum()
for column in CATEGORICAL_INPUT_COLUMNS:
    train_values = as_text(association_frame[column])
    test_values = as_text(association_test_frame[column])
    grouped = pd.DataFrame({'category': train_values, TARGET: association_frame[TARGET]}).groupby('category', dropna=False)[TARGET].agg(['count', 'mean'])
    between_group_ss = (grouped['count'] * (grouped['mean'] - association_frame[TARGET].mean()) ** 2).sum()
    categorical_rows.append({
        'feature': column, 'unique_categories': train_values.nunique(dropna=False),
        'smallest_category_count': grouped['count'].min(), 'categories_with_one_row': int((grouped['count'] == 1).sum()),
        'unseen_test_categories': len(set(test_values) - set(train_values)),
        'eta_squared_with_rent': between_group_ss / rent_variance_total if rent_variance_total else np.nan,
    })
categorical_target_associations = pd.DataFrame(categorical_rows).sort_values('eta_squared_with_rent', ascending=False)

display(numeric_target_associations)
display(numeric_feature_correlation.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1))
display(categorical_target_associations)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


,feature,pearson_with_rent,spearman_with_rent,non_null_rows
4,approval_date_num,0.534590,0.563916,150000
2,approval_year,0.517250,0.553775,150000
20,coe_quota_premium_mean,0.511271,0.510186,150000
6,flat_type_num,0.344918,0.342012,150000
0,FLOOR_AREA_SQM,0.329273,0.317703,150000
22,stock_close_mean,-0.013469,-0.293353,150000
10,YEAR_COMPLETED,0.078949,0.230557,150000
9,MAX_FLOOR,0.292075,0.228234,150000
1,LEASE_COMMENCE_DATE,0.229433,0.225456,150000
23,stock_volume_mean,-0.226463,-0.213468,150000


,FLOOR_AREA_SQM,LEASE_COMMENCE_DATE,approval_year,approval_month,approval_date_num,lease_age,flat_type_num,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,mrt_dist_nearest_km,mrt_within_1km,mrt_within_2km,mall_dist_nearest_km,mall_within_1km,mall_within_2km,school_dist_nearest_km,school_within_1km,school_within_2km,coe_quota_premium_mean,coe_bids_mean,stock_close_mean,stock_volume_mean,MONTHLY_RENT
FLOOR_AREA_SQM,1.000000,0.445808,-0.024617,-0.007124,-0.026035,-0.444643,0.945982,0.312179,0.006398,0.212884,0.454864,0.045045,-0.060245,-0.112802,-0.062166,0.062266,0.039424,-0.194804,0.222084,0.161299,-0.019498,0.000349,0.006042,0.011273,0.317703
LEASE_COMMENCE_DATE,0.445808,1.000000,0.028509,0.011683,0.031204,-0.992946,0.479778,0.446834,-0.017575,0.479111,0.991339,0.097071,-0.105436,-0.188287,-0.146119,0.138802,0.168995,-0.219135,0.230263,0.192871,0.026503,0.002313,-0.019727,-0.010963,0.225456
approval_year,-0.024617,0.028509,1.000000,-0.128415,0.973531,0.078449,-0.024615,-0.001781,-0.005535,0.022478,0.029405,0.007050,-0.000060,-0.001141,-0.003001,0.002192,0.003316,0.008225,-0.011017,-0.005596,0.663786,0.226720,-0.331124,-0.180245,0.553775
approval_month,-0.007124,0.011683,-0.128415,1.000000,0.091904,-0.026628,-0.006831,0.002916,0.007873,0.005478,0.010984,-0.002528,0.000712,0.006328,-0.000016,-0.000950,0.003426,0.000599,-0.002857,0.001288,0.268786,-0.278363,0.097526,-0.339097,0.063813
approval_date_num,-0.026035,0.031204,0.973531,0.091904,1.000000,0.072978,-0.025998,-0.000796,-0.003483,0.023732,0.031957,0.006539,0.000233,0.000174,-0.003009,0.002143,0.004173,0.008405,-0.011727,-0.005235,0.719217,0.179075,-0.297585,-0.255441,0.563916
lease_age,-0.444643,-0.992946,0.078449,-0.026628,0.072978,1.000000,-0.477200,-0.442089,0.017769,-0.478499,-0.984475,-0.095657,0.103243,0.184467,0.144379,-0.138010,-0.168867,0.218320,-0.229163,-0.192588,0.044861,0.022163,-0.016057,-0.008349,-0.165580
flat_type_num,0.945982,0.479778,-0.024615,-0.006831,-0.025998,-0.477200,1.000000,0.290285,0.005964,0.291278,0.488145,0.031377,-0.047550,-0.103352,-0.081962,0.081660,0.067039,-0.176987,0.208684,0.142752,-0.019880,0.000001,0.006093,0.010539,0.342012
LATITUDE,0.312179,0.446834,-0.001781,0.002916,-0.000796,-0.442089,0.290285,1.000000,0.052661,-0.070657,0.446332,0.207848,-0.229342,-0.469474,-0.050506,0.020541,-0.056752,-0.328925,0.503194,0.472926,0.002977,-0.001034,-0.006883,-0.000138,-0.078597
LONGITUDE,0.006398,-0.017575,-0.005535,0.007873,-0.003483,0.017769,0.005964,0.052661,1.000000,-0.063679,-0.012645,0.109246,-0.083601,-0.061424,-0.044254,0.085676,0.148200,-0.063237,0.158238,0.361825,0.000204,-0.003664,0.003990,0.000958,0.000909
MAX_FLOOR,0.212884,0.479111,0.022478,0.005478,0.023732,-0.478499,0.291278,-0.070657,-0.063679,1.000000,0.485064,0.028851,0.040284,0.059781,-0.101867,0.106368,0.159163,-0.003011,-0.083214,-0.083973,0.017295,0.003349,-0.010522,-0.007372,0.228234


,feature,unique_categories,smallest_category_count,categories_with_one_row,unseen_test_categories,eta_squared_with_rent
0,RENT_APPROVAL_DATE,51,2362,0,16,0.321478
3,STREET,576,2,0,0,0.144513
4,FLAT_TYPE,6,42,0,0,0.135229
2,BLOCK,2720,1,36,11,0.111891
5,FLAT_MODEL,20,1,1,0,0.064051
1,TOWN,26,438,0,0,0.037890
6,REGION,5,19043,0,0,0.014059


In [18]:
# Persist evidence only; no preprocessing or source-data file is changed by this notebook.
main_summary.to_csv(experiment_dir / 'exp_7_categorical_normalisation_summary.csv', index=False)
main_collisions.to_csv(experiment_dir / 'exp_7_categorical_collision_examples.csv', index=False)
coverage_report.to_csv(experiment_dir / 'exp_7_categorical_train_test_coverage.csv', index=False)
hdb_summary.to_csv(experiment_dir / 'exp_7_hdb_categorical_summary.csv', index=False)
hdb_collisions.to_csv(experiment_dir / 'exp_7_hdb_categorical_collision_examples.csv', index=False)
location_report.to_csv(experiment_dir / 'exp_7_location_key_coverage.csv', index=False)
area_report.to_csv(experiment_dir / 'exp_7_floor_area_sanity.csv', index=False)
outlier_summary.to_csv(experiment_dir / 'exp_7_numeric_outlier_summary.csv', index=False)
outlier_examples.to_csv(experiment_dir / 'exp_7_numeric_outlier_examples.csv', index=False)
numeric_target_associations.to_csv(experiment_dir / 'exp_7_numeric_target_associations.csv', index=False)
numeric_feature_correlation.to_csv(experiment_dir / 'exp_7_numeric_feature_correlation.csv')
categorical_target_associations.to_csv(experiment_dir / 'exp_7_categorical_target_associations.csv', index=False)

print('Audit complete. Review cleaning, outlier, and association reports before changing preprocessing.py.')
print('Use association measures to prioritise experiments, then confirm feature choices with chronological validation.')


Audit complete. Review cleaning, outlier, and association reports before changing preprocessing.py.
Use association measures to prioritise experiments, then confirm feature choices with chronological validation.
